# Gene sets round trip: GMT in, signature scores out

<a href="https://colab.research.google.com/github/theislab/cellucid-python/blob/main/docs/user_guide/r_package/f_notebooks_tutorials/11_gene_sets_round_trip.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

R sends gene sets to the viewer (`cellucid_push_gene_set()`), where
**Gene signature** and **Dot plot** offer them under **From
notebook**; the viewer's sets come back with `cellucid_gene_sets()`,
and the scores a gene signature computes come back as columns with
`cellucid_pull()`. GMT files carry sets between R, Python and the web
app.

**Time:** 10 minutes. The same tutorial in Python:
[Gene sets round trip](../../python_package/f_notebooks_tutorials/45_gene_sets_round_trip.ipynb).

In [ ]:
# A fresh R library (Google Colab's R runtime, a new machine) installs the packages once.
for (package in c("remotes", "SeuratObject")) {
  if (!requireNamespace(package, quietly = TRUE)) install.packages(package)
}
if (!requireNamespace("cellucid", quietly = TRUE)) {
  if ("cellucid" %in% rownames(available.packages())) install.packages("cellucid") else remotes::install_github("theislab/cellucid-r")
}

## The data

SeuratObject's own example, `pbmc_small`: 80 blood cells, 230 genes, a
t-SNE and the clusters `RNA_snn_res.1`. Use your own Seurat or
SingleCellExperiment object in its place.

In [ ]:
library(cellucid)
library(SeuratObject)
data("pbmc_small")
seu <- pbmc_small
seu

## Gene sets as a GMT file

In [ ]:
markers <- list(
  B = c("MS4A1", "CD79A", "CD79B"),
  T = c("CD3E", "CD3D", "IL7R"),
  NK = c("GNLY", "NKG7"),
  Mono = c("CD14", "LYZ", "S100A9")
)
cellucid_write_gmt(markers, "pbmc_markers.gmt", descriptions = list(B = "B cells"))
cat(readLines("pbmc_markers.gmt"), sep = "\n")
stopifnot(identical(cellucid_read_gmt("pbmc_markers.gmt"), markers))

## Into the viewer

In [ ]:
viewer <- cellucid_view(seu, dataset_id = "pbmc-small", dataset_name = "PBMC small")

In [ ]:
for (name in names(markers)) cellucid_push_gene_set(viewer, name, markers[[name]])

## Score them in the viewer

In **Analysis › Gene signature**, choose **From notebook**, pick `B`
and run it; do the same for `T`. Then run the next cell.

In [ ]:
pulled <- cellucid_pull(seu, viewer, return_summary = TRUE)
seu <- pulled$data
scores <- names(Filter(function(info) identical(info$origin, "analysis"), pulled$summary$columns))
scores
aggregate(seu[[]][, scores, drop = FALSE], by = list(groups = seu$groups), FUN = mean, na.rm = TRUE)

Each score is a numeric column named `<set> score`.
`cellucid_gene_sets(viewer)` lists every set the viewer holds — yours,
a marker genes result's (one per group) and a signature's — and
`cellucid_write_gmt()` writes them.

In [ ]:
held <- cellucid_gene_sets(viewer)
names(held)
cellucid_write_gmt(held, "from_viewer.gmt")

In [ ]:
cellucid_stop(viewer)

## Next

- [Subset and re-analyse](12_subset_and_reanalyse.ipynb)
- Python: [Gene sets round trip](../../python_package/f_notebooks_tutorials/45_gene_sets_round_trip.ipynb)